# Brute-Force CIEDE2000 Shade Matching

recommend_foundations ranks every product in the catalogue by CIEDE2000
distance from the skin estimate and returns the top matches: an exact,
brute-force nearest-neighbour search, with no clustering or approximate
candidate-restriction step. CIEDE2000 isn't a proper metric (it fails the
triangle inequality), so it isn't compatible with the spatial index
structures (KD-trees, ball-trees) that approximate nearest-neighbour
search normally relies on for speed — a full scan is the correct approach
here, not just the simplest one. This notebook measures how that full
scan's latency scales with catalogue size, to confirm it stays fast well
past the size of the real catalogue.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/YOUR_USERNAME/foundation-shade-recommender.git"


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return None


project_root = find_project_root(Path.cwd())

if project_root is None and "google.colab" in sys.modules:
    if "YOUR_USERNAME" in REPOSITORY_URL:
        raise ValueError(
            "Replace YOUR_USERNAME in REPOSITORY_URL after publishing the project to GitHub."
        )
    project_root = Path("/content/foundation-shade-recommender")
    if not project_root.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(project_root)], check=True)

if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

os.chdir(project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print("Project root:", project_root)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from foundation_matcher.data import load_foundation_catalog
from foundation_matcher.recommender import (
    benchmark_brute_force_search,
    recommend_foundations,
)
from foundation_matcher.visualization import plot_brute_force_latency


## 2. Rank shades for an example skin tone


In [ ]:
products = load_foundation_catalog()
example_skin_lab = np.array([65.0, 12.0, 18.0])

matches = recommend_foundations(example_skin_lab, products, top_n=5)
display(matches[["brand", "product", "hex", "color_distance"]])


## 3. Measure how search latency scales with catalogue size

The real catalogue is only a few hundred products. To see whether a full
CIEDE2000 scan would still be fast at a much larger scale, this resamples
the catalogue's LAB colours (with small jitter, so colours stay
realistic) up to each size below and times recommend_foundations over
random queries against it.


In [ ]:
latency = benchmark_brute_force_search(products, number_of_queries=50)
display(latency.round(6))

plot_brute_force_latency(latency)
plt.show()


## 4. Conclusion

Per-query latency stays well under the threshold that would make a user
wait, even at catalogue sizes far beyond what a real makeup brand
offers. Since brute force is both exact and fast at this scale, no
clustering or approximate-search step is needed — introducing one would
only risk excluding a genuinely close colour, for no measurable speed
benefit.
